In [22]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel
from IPython.display import Image
import operator

In [23]:
from dotenv import load_dotenv
load_dotenv("../../../LangChainModels/.env") # Load environment variables from .env file

True

In [24]:
generator_llm = ChatOpenAI(model="gpt-4o")
evaluator_llm = ChatOpenAI(model="gpt-4o-mini")
optimizer_llm = ChatOpenAI(model="gpt-4o")

In [25]:
#state
class State(TypedDict):
    topic: str 
    tweet : str 
    evaluation: Literal['approved', 'improve']
    feedback: str 
    attempt: int
    tweet_history : Annotated[list[str], operator.add]
    feedback_history : Annotated[list[str], operator.add]
    

In [26]:
def generate(state : State)-> State:
    topic = state['topic']
    attempt = state['attempt']
    # prompt
    prompt = [
    SystemMessage(content="You are a funny and clever Twitter/X influencer."),
    HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{topic}".

Rules:
- DO NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use genz slang and english for a genz punch to the tweet.
- This is version {attempt + 1}.
""")
]
    
    tweet = generator_llm.invoke(prompt).content
    
    return {
        'tweet' : tweet,
        'tweet_history': [tweet]
        
    }


In [27]:
class Evalutator(BaseModel):
    evaluation: Literal['approved', 'improve']
    feedback : str

structured_evaluation_llm = evaluator_llm.with_structured_output(Evalutator)

In [28]:
def evaluator(state: State)-> State:
    topic = state['topic']
    tweet = state['tweet']
    evaluator_prompt = [
    SystemMessage(
        content="""
You are a ruthless, no-laughs-given Twitter/X critic. You evaluate tweets based on humor,
originality, virality, Gen Z authenticity, and tweet format.

You are especially strict about whether Gen Z vocabulary and slang feel natural rather than
forced or outdated.
"""
    ),

    HumanMessage(
        content=f"""
Evaluate the following tweet:

Tweet:
"{tweet}"
on the topic : ''{topic}


Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?
2. Humor – Did it genuinely make you smile, laugh, or chuckle?
3. Punchiness – Is it tight, sharp, and scroll-stopping?
4. Virality Potential – Would people retweet, repost, quote-tweet, or share it?
5. Format – Is it a well-formed tweet, not a Q&A joke, and under 280 characters?
6. Gen Z Authenticity – Does it naturally sound like something a Gen Z person would
   actually post online?
7. Gen Z Vocabulary & Slang – Does it use Gen Z slang/vocabulary naturally where appropriate
   (e.g. "bro", "lowkey", "highkey", "fr", "ngl", "no cap", "it's giving", "cooked",
   "delulu", "ate", "slay", "mid", "vibe", "main character", "rent free", "POV", etc.)?

Do NOT require slang in every tweet. Penalize slang if it feels forced, excessive,
cringe, outdated, or inserted just to sound Gen Z.

Auto-reject if:
- It's written in question-answer format (e.g. "Why did..." or "What happens when...")
- It exceeds 280 characters.
- It feels like a traditional setup-punchline joke.
- It lacks a clear punchline, humorous observation, or relatable take.
- It uses generic, throwaway, or deflating lines that weaken the humor
  (e.g. "Masterpieces of the auntie-uncle universe" or vague summaries).
- The Gen Z slang feels forced, unnatural, or disconnected from the joke.
- It sounds overly formal, corporate, or like an AI-generated attempt at sounding Gen Z.

### Respond ONLY in this structured format:

- evaluation: "approved" or "improve"
- feedback: One paragraph explaining the tweet's strengths and weaknesses,
  including specific suggestions for improving its humor, punchiness,
  virality, and Gen Z authenticity.
"""
    )
]

    response = structured_evaluation_llm.invoke(evaluator_prompt)
    return {
        'evaluation': response.evaluation,
        'feedback': response.feedback,
        'feedback_history' : [response.feedback]
    }

In [29]:
def decide_next(state : State) -> State:
    if state['evaluation']== 'approved':
        return 'end'
    if state['attempt'] >= 3:
        return 'end'
    return 'retry'

In [30]:
def optimizer(state : State)-> State:
    tweet = state['tweet']
    feedback = state['feedback']
    topic = state['topic']
    attempt = state['attempt']
    optimizer_prompt = [
    SystemMessage(
        content="""
You are an expert Twitter/X copywriter who specializes in viral Gen Z humor.

Your job is to improve an existing tweet based on critical feedback.
Do not blindly rewrite it. Preserve anything that is already funny or effective,
while fixing the weaknesses identified by the evaluator.

The final tweet should feel natural, witty, relatable, and genuinely written
by a Gen Z person — NOT like an AI trying to sound Gen Z.
"""
    ),

    HumanMessage(
        content=f"""
Improve the following tweet based on the evaluator's feedback.

Topic:
"{topic}"

Current Tweet:
"{tweet}"

Evaluator Feedback:
"{feedback}"

Requirements for the improved tweet:

1. Keep it relevant to the topic.
2. Make it genuinely funny, clever, and scroll-stopping.
3. Strengthen the punchline and comedic timing.
4. Make it concise and impactful.
5. Keep it under 280 characters.
6. Do NOT use a question-answer format.
7. Do NOT turn it into a traditional setup-punchline joke.
8. Use observational humor, irony, sarcasm, meme logic, or relatable situations.
9. Use Gen Z vocabulary and slang naturally when it improves the joke.
10. Gen Z slang should NEVER feel forced, excessive, outdated, or cringe.
11. Keep the language casual, conversational, and authentic.
12. Do not add unnecessary explanations, hashtags, or generic closing lines.
13. Do not simply paraphrase the original tweet — make a meaningful improvement
    based on the feedback.

Return ONLY the improved tweet.
"""
    )
]
    tweet = optimizer_llm.invoke(optimizer_prompt).content
    return {
        'tweet': tweet,
        'attempt': attempt + 1,
        'tweet_history': [tweet]
    }

In [31]:
graph = StateGraph(State)

In [32]:
graph.add_node('generate_tweet', generate)
graph.add_node('evaluate_tweet', evaluator)
graph.add_node('optimize_tweet', optimizer)

In [33]:
graph.add_edge(START, 'generate_tweet')
graph.add_edge('generate_tweet', 'evaluate_tweet')
graph.add_conditional_edges('evaluate_tweet', decide_next, {
    'end': END,
    'retry': 'optimize_tweet'
})
graph.add_edge('optimize_tweet', 'evaluate_tweet')

In [34]:
app = graph.compile()

In [35]:
topic = input('enter the topic you want to generate tweet on')
result = app.invoke({
    "attempt": 0,
    'topic': topic,
    'tweet': '',
    "evaluation": "",
    "feedback": "",
    
})
print(result)

{'topic': 'foiujbf', 'tweet': '\nJust typed "foiujbf," and now my autocorrect is in therapy. Today’s typing: 404 brain not found. 💀🎉', 'evaluation': 'improve', 'feedback': "While the tweet attempts to balance humor and relatability with a quirky typing mishap, it falls flat in several areas. The phrase '404 brain not found' is a bit of a cliché and has been used frequently in similar contexts, diminishing its originality. The humor is light and could bring a smirk, but it lacks a sharp punch that would prompt a real laugh, making it feel somewhat bloated and unfocused. The emojis are fine but contribute little to adding humor. For a stronger impact, consider tightening the phrasing for punchiness, perhaps using a more unique take on typing blunders or clarifying the autocorrect therapy bit. Additionally, integrating authentic Gen Z slang more fluidly could enhance its appeal; words like 'cooked' or 'delulu' might really amplify its vibe if aligned with the context.", 'attempt': 3, 'twe